In [207]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [208]:
df = pd.read_csv('/Users/sanjayduduka/Desktop/AI-Based Hiring Prediction System/Dataset/AI-Based Hiring Prediction System (1).csv')
df.head()

,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
0,1,Ashley Ali,"TensorFlow, NLP, Pytorch",10,B.Sc,NaN,AI Researcher,Hire,104895,8,100
1,2,Wesley Roman,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,Hire,113002,1,100
2,3,Corey Sanchez,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,Hire,71766,7,70
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,Hire,46848,0,95
4,5,Julie Hill,"SQL, React, Java",4,PhD,NaN,Software Engineer,Hire,87441,9,100


In [209]:
df.shape

(1000, 11)

In [210]:
df.size

11000

In [211]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Resume_ID               1000 non-null   int64
 1   Name                    1000 non-null   str  
 2   Skills                  1000 non-null   str  
 3   Experience (Years)      1000 non-null   int64
 4   Education               1000 non-null   str  
 5   Certifications          726 non-null    str  
 6   Job Role                1000 non-null   str  
 7   Recruiter Decision      1000 non-null   str  
 8   Salary Expectation ($)  1000 non-null   int64
 9   Projects Count          1000 non-null   int64
 10  AI Score (0-100)        1000 non-null   int64
dtypes: int64(5), str(6)
memory usage: 86.1 KB


In [212]:
df.describe()

,Resume_ID,Experience (Years),Salary Expectation ($),Projects Count,AI Score (0-100)
count,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000
mean,500.500000,4.896000,79994.486000,5.13300,83.950000
std,288.819436,3.112695,23048.472549,3.23137,20.983036
min,1.000000,0.000000,40085.000000,0.00000,15.000000
25%,250.750000,2.000000,60415.750000,2.00000,70.000000
50%,500.500000,5.000000,79834.500000,5.00000,100.000000
75%,750.250000,8.000000,99583.250000,8.00000,100.000000
max,1000.000000,10.000000,119901.000000,10.00000,100.000000


In [213]:
df.drop('Resume_ID' , axis=1 , inplace=True)
df.drop('Name' , axis=1 , inplace=True)

In [214]:
df.isnull().sum().sum()

np.int64(274)

In [215]:
df.select_dtypes(include='number').isnull().sum()

Experience (Years)        0
Salary Expectation ($)    0
Projects Count            0
AI Score (0-100)          0
dtype: int64

In [216]:
df.select_dtypes(exclude='number').isnull().sum()

Skills                  0
Education               0
Certifications        274
Job Role                0
Recruiter Decision      0
dtype: int64

In [217]:
df["Certifications"] = df["Certifications"].fillna("No")

In [218]:
df.select_dtypes(exclude='number').isnull().sum()

Skills                0
Education             0
Certifications        0
Job Role              0
Recruiter Decision    0
dtype: int64

In [219]:
df["Recruiter Decision"].value_counts(normalize=True) * 100

Recruiter Decision
Hire      81.2
Reject    18.8
Name: proportion, dtype: float64

In [220]:
df["Recruiter Decision"] = df["Recruiter Decision"].map({
    "Hire": 1,
    "Reject": 0
})
df["Recruiter Decision"]

0      1
1      1
2      1
3      1
4      1
      ..
995    0
996    0
997    1
998    1
999    1
Name: Recruiter Decision, Length: 1000, dtype: int64

In [221]:
df['Recruiter Decision'].value_counts()

Recruiter Decision
1    812
0    188
Name: count, dtype: int64

In [222]:
X= df.drop('Recruiter Decision', axis=1)
Y= df['Recruiter Decision']

In [223]:
X_train ,X_test , Y_train,Y_test =train_test_split(
    X, Y , test_size=0.2,
    random_state=42,
    stratify=Y
)

In [224]:
text_feature = "Skills"

numeric_features = X.select_dtypes(include='number').columns

categorical_features = X.select_dtypes(exclude='number').columns

In [225]:
preprocessor = ColumnTransformer(
    transformers=[
        
        (
            "text",
            TfidfVectorizer(
                max_features=100,
                ngram_range=(1,2)
            ),
            text_feature
        ),

        (
            "num",
            StandardScaler(),
            numeric_features
        ),

        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [226]:
models = {

    "Logistic Regression":
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Random Forest":
    RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost":
    XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight = len(Y_train[Y_train==0]) / len(Y_train[Y_train==1])
    ),

    "LightGBM":
    LGBMClassifier(
        class_weight="balanced",
        random_state=42,
        verbose=-1
    ),

    "CatBoost":
    CatBoostClassifier(
        auto_class_weights="Balanced",
        random_state=42,
        verbose=0
    )
}

In [232]:
from imblearn.pipeline import Pipeline

pipelines = {}

for name, model in models.items():

    pipelines[name] = Pipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])

In [ ]:
results = []

for name, pipe in pipelines.items():

    pipe = ImbPipeline([
        ("preprocessor", pipe.named_steps["preprocessor"]),
        ("smote", SMOTE(random_state=42)),
        ("model", pipe.named_steps["model"]),
    ])
    pipe.fit(X_train, Y_train)

    y_pred = pipe.predict(X_test)

    y_prob = pipe.predict_proba(X_test)[:,1]

    results.append({

        "Model": name,

        "Accuracy": accuracy_score(Y_test, y_pred),

        "Precision": precision_score(Y_test, y_pred),

        "Recall": recall_score(Y_test, y_pred),

        "F1": f1_score(Y_test, y_pred),

        "ROC_AUC": roc_auc_score(Y_test, y_prob)

    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by="ROC_AUC",
    ascending=False
)

TypeError: All intermediate steps should be transformers and implement fit and transform or be the string 'passthrough' 'SMOTE(random_state=42)' (type <class 'imblearn.over_sampling._smote.base.SMOTE'>) doesn't